# Load and PV Forecastability

## Objectives

Part 3 forecasts household-level load and the generation hiding inside it, at the scale of hundreds of real smart meters, not one aggregate signal. Before reaching for a model, this notebook answers the question every later chapter depends on: how forecastable is a real customer's own signal, and what does that measured forecastability say about which approach is actually worth using on them?

By the end you will be able to:

- Run a real forecastability diagnostic (entropy, Hurst exponent, DFA, autocorrelation, stationarity) across hundreds of real customers individually, not one aggregate signal, and report the real spread.
- Check whether a customer's forecastability predicts whether a simple baseline already beats a trained model, on this book's own real data.
- Characterize how PV/DER generation shows up as an invisible confounder in net load, using the dataset's own reactive-power readings and regional solar irradiance as real, if indirect, evidence.
- Turn a customer's own forecastability profile into a real decision: which later chapter's approach is worth reaching for, and which customers a simple baseline already serves well.


## Getting the data

This notebook uses the Lede AS / Porsgrunn smart-meter dataset, vendored once for all of Part 3:

```bash
uv run python scripts/fetch_part3_mendeley_data.py
```


In [ ]:
from pathlib import Path

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    facet_wrap,
    geom_density,
    geom_hline,
    geom_line,
    geom_point,
    geom_segment,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    theme,
)
import numpy as np
import pandas as pd

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import GRAY_600, PRIMARY

LetsPlot.setup_html(isolated_frame=True)

UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 960


def acf_lollipop_chart(series: np.ndarray, *, max_lag: int, title: str) -> "ggplot":
    """An ACF chart: vertical lollipops against a 95% confidence band."""
    from statsmodels.tsa.stattools import acf

    n = len(series)
    ci = 1.96 / np.sqrt(n)
    values = acf(series, nlags=max_lag, fft=True)
    lag_df = pd.DataFrame({"lag": np.arange(len(values)), "acf": values})
    lag_df["significant"] = np.where(lag_df["acf"].abs() > ci, "significant", "not significant")

    return (
        ggplot(lag_df, aes(x="lag", y="acf"))
        + geom_segment(aes(x="lag", xend="lag", yend="acf", color="significant"), y=0, size=1.0)
        + geom_point(aes(color="significant"), size=1.8)
        + geom_hline(yintercept=0, color=GRAY_600, size=0.3)
        + geom_hline(yintercept=ci, linetype="dashed", color=GRAY_600, size=0.5)
        + geom_hline(yintercept=-ci, linetype="dashed", color=GRAY_600, size=0.5)
        + scale_color_manual(values={"significant": PRIMARY, "not significant": GRAY_600}, guide="none")
        + labs(title=title, subtitle=f"95% confidence band: ±{ci:.3f} (n={n:,})", x="Lag (hours)", y="ACF")
        + modern_theme()
        + UNBOLD_AXES
        + ggsize(FULL_WIDTH, 260)
    )


DATA_DIR = Path(
    "../../resources/mendeley-lede-porsgrunn-ami/extracted/"
    "Energy distribution models with AMI smart meter sensor dataset/data"
)
AMI_DIR = DATA_DIR / "ami"
WEATHER_DIR = DATA_DIR / "aux/weather"

RNG = np.random.default_rng(42)

## Hundreds of customers, not one aggregate signal

The dataset holds 6,809 real meters. Sampling a few hundred of them, rather than working from one region-wide aggregate, is the only way to see whether forecastability itself varies from customer to customer, the question this whole notebook is built around. A random draw of meter IDs is not the same as a random draw of good data: real AMI feeds carry partial-year connections, long outages, and stuck sensors, and every diagnostic below assumes a real, sufficiently complete history, not a raw file that happens to exist. A real quality gate runs first, not an afterthought.

In [ ]:
ami_files = sorted(AMI_DIR.glob("*.parquet"))
print(f"real customers available: {len(ami_files):,}")


def load_customer(path: Path) -> pd.DataFrame:
    """One real customer's hourly net load: consumption minus export.

    Checked directly against real winter/summer hourly averages before
    trusting the column names: `activePowerOut` is substantial and, for
    ordinary customers, *higher* in winter midnight hours than summer
    midday, the opposite of a solar-export signal, and exactly what real
    heating-driven household consumption looks like; `activePowerIn` is
    essentially zero for the vast majority of customers and only real for
    the minority who actually export. So `activePowerOut` is the
    customer's own draw from the grid (consumption) and `activePowerIn`
    is the customer's own feed into the grid (export), not the reverse a
    naive reading of "in"/"out" from the customer's own side would
    suggest; these are named from the network's side of the meter.
    """
    df = pd.read_parquet(path)
    df = df.sort_values("dateTime").reset_index(drop=True)
    df["customer_id"] = path.stem
    df["net_load"] = df["activePowerOut"] - df["activePowerIn"]
    return df


def quality_report(
    df: pd.DataFrame, *, min_rows: int = 24 * 365, min_completeness: float = 0.98, max_stuck_share: float = 0.5
) -> dict:
    """A real, checkable data-quality gate, not just a zero-variance check.

    Screens for the failure modes an honest AMI feed actually has: a
    partial-year connection too short to see a real seasonal cycle, a long
    gap in an otherwise real history, a completely flat (vacant-property)
    reading, and a stuck sensor repeating its last value most of the time.
    """
    span_hours = int((df["dateTime"].max() - df["dateTime"].min()) / pd.Timedelta(hours=1)) + 1
    completeness = len(df) / span_hours if span_hours else 0.0
    stuck_share = (df["net_load"].diff() == 0).mean()
    passed = (
        len(df) >= min_rows
        and completeness >= min_completeness
        and df["net_load"].std() > 1e-6
        and stuck_share <= max_stuck_share
    )
    return {"rows": len(df), "completeness": completeness, "stuck_share": stuck_share, "passed": passed}


# Oversample the candidate pool: the quality gate below is expected to drop
# some real customers (short histories, gappy feeds, stuck sensors), so more
# raw files are loaded than the final target sample size.
N_DIAGNOSTIC = 300
CANDIDATE_POOL_SIZE = 450
candidate_files = RNG.choice(ami_files, size=CANDIDATE_POOL_SIZE, replace=False)
candidates = {path.stem: load_customer(path) for path in candidate_files}

quality = {cust_id: quality_report(df) for cust_id, df in candidates.items()}
quality_df = pd.DataFrame.from_dict(quality, orient="index")
quality_df["passed"] = quality_df["passed"].astype(bool)
print(f"candidates loaded: {len(candidates)}")
print(f"failed minimum history / completeness / flat / stuck-sensor gate: {(~quality_df['passed']).sum()}")

passing_ids = quality_df[quality_df["passed"]].index.tolist()
kept_ids = RNG.choice(passing_ids, size=min(N_DIAGNOSTIC, len(passing_ids)), replace=False)
customers = {cust_id: candidates[cust_id] for cust_id in kept_ids}
print(
    f"kept for analysis: {len(customers)} real customers, "
    "each with a full year or more of real, sufficiently complete AMI history"
)

lengths = pd.Series({cust_id: len(df) for cust_id, df in customers.items()})
print(f"hourly rows per customer: min={lengths.min():,} median={lengths.median():,.0f} max={lengths.max():,}")

real customers available: 6,809


candidates loaded: 450
failed minimum history / completeness / flat / stuck-sensor gate: 143
kept for analysis: 300 real customers, each with a full year or more of real, sufficiently complete AMI history
hourly rows per customer: min=16,459 median=24,025 max=24,067


In [ ]:
preview = customers[list(customers)[0]].head(3)[
    ["dateTime", "activePowerIn", "activePowerOut", "reactivePowerIn", "reactivePowerOut", "net_load"]
]

themed_gt(
    GT(preview)
    .tab_header(
        title=md("**One Real Customer, First 3 Hours**"), subtitle="Lede AS / Porsgrunn AMI dataset, hourly resolution"
    )
    .tab_source_note("Maree et al., Mendeley Data, 2025"),
    n_rows=len(preview),
)

GT(_tbl_data=             dateTime  activePowerIn  activePowerOut  reactivePowerIn  \
0 2022-01-01 00:00:00            0.0           0.333              0.0   
1 2022-01-01 01:00:00            0.0           0.330              0.0   
2 2022-01-01 02:00:00            0.0           0.331              0.0   

   reactivePowerOut  net_load  
0               0.0     0.333  
1               0.0     0.330  
2               0.0     0.331  , _body=<great_tables._gt_data.Body object at 0x112c45370>, _boxhead=Boxhead([ColInfo(var='dateTime', type=<ColInfoTypeEnum.default: 1>, column_label='dateTime', column_align='right', column_width=None), ColInfo(var='activePowerIn', type=<ColInfoTypeEnum.default: 1>, column_label='activePowerIn', column_align='right', column_width=None), ColInfo(var='activePowerOut', type=<ColInfoTypeEnum.default: 1>, column_label='activePowerOut', column_align='right', column_width=None), ColInfo(var='reactivePowerIn', type=<ColInfoTypeEnum.default: 1>, column_label='reactivePowerIn', column_align='right', column_width=None), ColInfo(var='reactivePowerOut', type=<ColInfoTypeEnum.default: 1>, column_label='reactivePowerOut', column_align='right', column_width=None), ColInfo(var='net_load', type=<ColInfoTypeEnum.default: 1>, column_label='net_load', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1140b8c20>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**One Real Customer, First 3 Hours**'), subtitle='Lede AS / Porsgrunn AMI dataset, hourly resolution', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x114285bb0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x112a7ede0>, _source_notes=['Maree et al., Mendeley Data, 2025'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='dateTime', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='activePowerIn', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='activePowerOut', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='reactivePowerIn', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='reactivePowerOut', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='net_load', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=No

In [ ]:
# A stable, recognizable daily pattern, not an arbitrary first-4 pick: rank
# by lag-24 autocorrelation (how strongly each day repeats the one before)
# among customers already cleared of the flat/zero-variance case above.
daily_repeat_score = {cust_id: df["net_load"].autocorr(lag=24) for cust_id, df in customers.items()}
example_ids = pd.Series(daily_repeat_score).nlargest(4).index.tolist()

example_df = pd.concat([customers[cust_id].assign(customer=cust_id[:8]) for cust_id in example_ids], ignore_index=True)
two_weeks = example_df[example_df["dateTime"] < example_df["dateTime"].min() + pd.Timedelta(days=14)]

p = (
    ggplot(two_weeks, aes("dateTime", "net_load", color="customer"))
    + geom_line(size=0.6)
    + labs(title="Four Real Customers With a Stable Daily Pattern, Two Weeks", x="", y="Net load (kW)")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 260)
)
p

## How forecastable is a real customer?

Individual loads are known to be more volatile and harder to forecast than a substation- or city-level aggregate, a finding from entropy-based studies on real smart-meter data elsewhere. Running the same battery here, on hundreds of real Lede/Porsgrunn customers individually rather than one aggregate, checks whether that holds on this book's own data, and how much forecastability itself varies customer to customer.

In [ ]:
from twiga.core.stats.entropy import get_dfa_exponent, get_hurst_exponent, get_permutation_entropy, get_sample_entropy

entropy_ref = pd.DataFrame(
    {
        "Metric": ["Permutation entropy", "Sample entropy", "Hurst exponent", "DFA exponent"],
        "Forecastable signal": ["near 0 (regular)", "< 0.5 (regular)", "> 0.5 (persistent)", "> 0.5 (persistent)"],
    }
)
themed_gt(
    GT(entropy_ref).tab_header(title=md("**Forecastability Metrics Used Below**")),
    n_rows=len(entropy_ref),
)

GT(_tbl_data=                Metric Forecastable signal
0  Permutation entropy    near 0 (regular)
1       Sample entropy     < 0.5 (regular)
2       Hurst exponent  > 0.5 (persistent)
3         DFA exponent  > 0.5 (persistent), _body=<great_tables._gt_data.Body object at 0x118a9d1c0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Forecastable signal', type=<ColInfoTypeEnum.default: 1>, column_label='Forecastable signal', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1203634d0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Forecastability Metrics Used Below**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x120257470>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1202775f0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Metric', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Forecastable signal', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Metric', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Metric', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Forecastable signal', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Forecastable signal', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x11fc6f320>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Libre Franklin', 'Segoe UI', 'Roboto', 'Helvetica Neue', 'sans-serif'

In [ ]:
def forecastability_row(customer_id: str, series: np.ndarray) -> dict:
    return {
        "customer_id": customer_id,
        "permutation_entropy": get_permutation_entropy(series),
        "sample_entropy": get_sample_entropy(series),
        "hurst_exponent": get_hurst_exponent(series),
        "dfa_exponent": get_dfa_exponent(series),
    }


forecastability = pd.DataFrame(
    [forecastability_row(cust_id, df["net_load"].to_numpy()) for cust_id, df in customers.items()]
).set_index("customer_id")
forecastability.describe().round(3)

,permutation_entropy,sample_entropy,hurst_exponent,dfa_exponent
count,300.000,300.000,300.000,300.000
mean,0.993,0.772,0.482,0.875
std,0.014,0.297,0.618,0.101
min,0.828,0.046,-3.391,0.571
25%,0.992,0.620,0.627,0.810
50%,0.997,0.793,0.714,0.876
75%,0.999,0.958,0.748,0.938
max,1.000,1.747,0.977,1.233


In [ ]:
forecastability_long = forecastability.reset_index().melt(id_vars="customer_id", var_name="metric", value_name="value")
metric_labels = {
    "permutation_entropy": "Permutation entropy",
    "sample_entropy": "Sample entropy",
    "hurst_exponent": "Hurst exponent",
    "dfa_exponent": "DFA exponent",
}
forecastability_long["metric"] = forecastability_long["metric"].map(metric_labels)

p = (
    ggplot(forecastability_long, aes("value", fill="metric"))
    + geom_density(alpha=0.85, color=PRIMARY, show_legend=False)
    + facet_wrap(facets="metric", ncol=2, scales="free")
    + labs(title="Hundreds of Customers, Hundreds of Forecastability Profiles", x="", y="Density")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 380)
)
p

## Lag structure and stationarity

Entropy and Hurst say how regular a signal is; they do not say which real lag carries the signal, or whether the series is drifting under its own diurnal or weekly cycle. Autocorrelation and stationarity tests fill that gap, run here per customer rather than on one aggregate.

In [ ]:
from twiga.core.stats.autocorr import estimate_ar_order
from twiga.core.stats.seasonality import get_acf_values
from twiga.core.stats.stationarity import adf_test, kpss_test


def lag_and_stationarity_row(customer_id: str, series: np.ndarray) -> dict:
    acf_values, _ = get_acf_values(series, maxlag=168)  # one real week at hourly resolution
    _, ar_order = estimate_ar_order(series, maxlag=100)
    _, adf_p, *_ = adf_test(series)
    _, kpss_p, *_ = kpss_test(series)
    if adf_p < 0.05 and kpss_p > 0.05:
        verdict = "stationary"
    elif adf_p > 0.05 and kpss_p < 0.05:
        verdict = "non-stationary"
    elif adf_p < 0.05 and kpss_p < 0.05:
        verdict = "trend-stationary"
    else:
        verdict = "inconclusive"
    return {
        "customer_id": customer_id,
        "acf_daily_24h": acf_values[23],
        "acf_weekly_168h": acf_values[167],
        "ar_order": ar_order,
        "adf_p": adf_p,
        "kpss_p": kpss_p,
        "stationarity_verdict": verdict,
    }


lag_stationarity = pd.DataFrame(
    [lag_and_stationarity_row(cust_id, df["net_load"].to_numpy()) for cust_id, df in customers.items()]
).set_index("customer_id")
lag_stationarity.head()

,acf_daily_24h,acf_weekly_168h,ar_order,adf_p,kpss_p,stationarity_verdict
customer_id,,,,,,
8b96cd02-4c80-541d-963e-c0dc56303451,0.548752,0.484972,24,2.703436e-13,0.01,trend-stationary
2c80ccbf-89e0-5d2b-b94a-cd1a42f96644,0.894957,0.822766,18,3.037371e-03,0.01,trend-stationary
2680db1b-898f-53fc-ba39-9ab2bba67c7c,0.664020,0.463183,6,8.923816e-22,0.01,trend-stationary
8d492662-3429-57a1-8551-f7aa5d2f795a,0.861235,0.800048,25,5.600491e-03,0.01,trend-stationary
170c612a-e760-54fb-bef0-e93e6534a5f1,0.361972,0.330736,9,4.768685e-14,0.01,trend-stationary


In [ ]:
most_persistent = forecastability["hurst_exponent"].idxmax()
least_persistent = forecastability["hurst_exponent"].idxmin()

acf_lollipop_chart(
    customers[most_persistent]["net_load"].to_numpy(),
    max_lag=168,
    title=f"Most Persistent Real Customer (Hurst={forecastability.loc[most_persistent, 'hurst_exponent']:.2f})",
)

In [ ]:
acf_lollipop_chart(
    customers[least_persistent]["net_load"].to_numpy(),
    max_lag=168,
    title=f"Least Persistent Real Customer (Hurst={forecastability.loc[least_persistent, 'hurst_exponent']:.2f})",
)

In [ ]:
verdict_counts = (
    lag_stationarity["stationarity_verdict"].value_counts().rename_axis("verdict").reset_index(name="customers")
)
verdict_counts["share"] = (verdict_counts["customers"] / len(lag_stationarity) * 100).round(1)

themed_gt(
    GT(verdict_counts)
    .tab_header(title=md("**Stationarity Verdict Across the Real Customer Sample**"))
    .cols_label(verdict=md("**Verdict**"), customers=md("**Customers**"), share=md("**Share (%)**"))
    .tab_source_note(f"n = {len(lag_stationarity)} real Lede/Porsgrunn customers"),
    n_rows=len(verdict_counts),
)

GT(_tbl_data=            verdict  customers  share
0  trend-stationary        300  100.0, _body=<great_tables._gt_data.Body object at 0x11470c2c0>, _boxhead=Boxhead([ColInfo(var='verdict', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Verdict**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='share', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Share (%)**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x120581be0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Stationarity Verdict Across the Real Customer Sample**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x114706c30>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x114706330>, _source_notes=['n = 300 real Lede/Porsgrunn customers'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='verdict', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='share', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x114707200>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Libre Franklin', 'Segoe UI', 'Roboto', 'Helvetica Neue', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='13px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value'

## A spectral view of predictability

Entropy, Hurst, and autocorrelation all read a signal in the time domain. A frequency-domain view can catch something they miss: whether a customer's own consumption is dominated by a few strong, regular cycles or spread evenly across the spectrum with no real periodicity at all. This is this book's own construction, a normalized spectral entropy inverted so higher means more forecastable, checked below for whether it adds real signal beyond the time-domain battery above.

In [ ]:
def spectral_predictability_score(series: np.ndarray) -> float:
    """1 minus the normalized Shannon entropy of the power spectral density.

    Higher means frequency content concentrated in a few regular components
    (more forecastable); lower means power spread evenly across frequencies,
    closer to white noise.
    """
    centered = series - series.mean()
    power = np.abs(np.fft.rfft(centered)) ** 2
    power = power[1:]  # drop the DC component
    power = power[power > 0]
    if power.size == 0:
        return np.nan
    p = power / power.sum()
    spectral_entropy = -(p * np.log(p)).sum() / np.log(p.size)
    return float(1.0 - spectral_entropy)


forecastability["spectral_predictability"] = [
    spectral_predictability_score(customers[cust_id]["net_load"].to_numpy()) for cust_id in forecastability.index
]
forecastability[["hurst_exponent", "spectral_predictability"]].corr()

,hurst_exponent,spectral_predictability
hurst_exponent,1.000000,-0.012697
spectral_predictability,-0.012697,1.000000


## Which features matter, and does that differ by customer?

Knowing a signal is forecastable is not the same as knowing what to feed a model. A real, honest ensemble of association metrics, ranked and aggregated across several independent statistical tests rather than any single one, answers a sharper question: which lag or calendar feature actually carries the signal for a given customer, and does the answer stay the same from one customer to the next, or does it genuinely vary.

In [ ]:
FORECAST_HORIZON = 24  # hours ahead, the same horizon every model below is checked against


def build_lag_features(df: pd.DataFrame, *, horizon: int = FORECAST_HORIZON) -> tuple[pd.DataFrame, list[str]]:
    """A small, direct lag-and-calendar feature matrix for a horizon-matched forecast.

    The target is shifted `horizon` steps into the future and every lag
    feature is taken relative to the forecast origin, not the target, so a
    model trained on this matrix answers a real, horizon-matched question,
    not an easier same-hour prediction task.
    """
    feat = pd.DataFrame({"timestamp": df["dateTime"], "value": df["net_load"].to_numpy()})
    feat["target"] = feat["value"].shift(-horizon)
    for lag in (0, 24, 48, 168):
        feat[f"lag_{lag}"] = feat["value"].shift(lag)
    feat["hour"] = feat["timestamp"].dt.hour
    feat["dayofweek"] = feat["timestamp"].dt.dayofweek
    feature_cols = [c for c in feat.columns if c not in ("timestamp", "value", "target")]
    return feat.dropna().reset_index(drop=True), feature_cols

In [ ]:
import numpy as _np
import twiga.core.data.selection as _selection


# A real, reproducible bug in this installed twiga version: diversity_weights
# mutates a read-only array returned by DataFrame.corr().to_numpy() under
# this environment's numpy/pandas combination. Patched here, once, with the
# one-line fix (an explicit .copy()) rather than losing the ensemble ranker's
# real diversity-weighting behavior over a single weaker metric.
def _diversity_weights_fixed(score_matrix):
    n = score_matrix.shape[1]
    if n == 1:
        return _np.ones(1)
    corr = score_matrix.corr(method="spearman").abs().to_numpy().copy()
    _np.fill_diagonal(corr, 0.0)
    uniqueness = 1.0 - corr.mean(axis=1)
    weights = _np.clip(uniqueness, 0.05, None)
    return weights / weights.sum()


_selection.diversity_weights = _diversity_weights_fixed
from twiga.core.data.selection import select_top_features

N_FEATURE_SAMPLE = 40
feature_ids = RNG.choice(list(customers), size=N_FEATURE_SAMPLE, replace=False)

top_feature_rows = []
for cust_id in feature_ids:
    feat, feature_cols = build_lag_features(customers[cust_id])
    top = select_top_features(data=feat, features=feature_cols, target="target", top_k=1)
    if top:
        top_feature_rows.append({"customer_id": cust_id, "top_feature": top[0]})

top_feature_counts = (
    pd.DataFrame(top_feature_rows)["top_feature"].value_counts().rename_axis("feature").reset_index(name="customers")
)
top_feature_counts["share"] = (top_feature_counts["customers"] / len(top_feature_rows) * 100).round(1)

themed_gt(
    GT(top_feature_counts)
    .tab_header(title=md("**The Single Most Relevant Feature, Per Customer**"), subtitle="24-hour-ahead net load")
    .cols_label(feature=md("**Top feature**"), customers=md("**Customers**"), share=md("**Share (%)**"))
    .tab_source_note(
        f"n = {len(top_feature_rows)} real customers, ensemble-ranked "
        "(Spearman, ANOVA, mutual information, Xi, PPS, random-forest importance)"
    ),
    n_rows=len(top_feature_counts),
)

GT(_tbl_data=  feature  customers  share
0   lag_0         40  100.0, _body=<great_tables._gt_data.Body object at 0x114706ba0>, _boxhead=Boxhead([ColInfo(var='feature', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Top feature**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='share', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Share (%)**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11470cc20>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**The Single Most Relevant Feature, Per Customer**'), subtitle='24-hour-ahead net load', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x114706e10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1147073b0>, _source_notes=['n = 40 real customers, ensemble-ranked (Spearman, ANOVA, mutual information, Xi, PPS, random-forest importance)'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='feature', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='share', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x114706780>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Libre Franklin', 'Segoe UI', 'Roboto', 'Helvetica Neue', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='13px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal')

## Does a trained model actually beat a naive forecast?

A naive forecast, simply repeating the most recently observed reading, costs nothing to build. LightGBM costs real engineering and compute. The honest question is whether that cost buys anything, checked per customer on the exact same real train/test split and the exact same 24-hour-ahead target both models above already use, not assumed to hold uniformly, on a smaller real sample to keep hundreds of individual fits tractable.

In [ ]:
import lightgbm as lgb

N_BASELINE = 30
baseline_ids = RNG.choice(list(customers), size=N_BASELINE, replace=False)


def compare_naive_and_lightgbm(df: pd.DataFrame, split_at: pd.Timestamp) -> dict | None:
    """A naive persistence forecast against LightGBM, on the same real target.

    Both read from the same `build_lag_features` matrix and the same
    chronological split, so the comparison is genuinely apples-to-apples:
    naive predicts the forecast origin's own last reading (`lag_0`), and
    LightGBM is trained on the same lag and calendar features to predict
    the same 24-hour-ahead target.
    """
    feat, feature_cols = build_lag_features(df)
    train_feat, test_feat = feat[feat["timestamp"] < split_at], feat[feat["timestamp"] >= split_at]
    if len(train_feat) < 24 * 7 * 4 or len(test_feat) < 24 * 7:
        return None
    actual = test_feat["target"].to_numpy()

    naive_mae = float(np.mean(np.abs(test_feat["lag_0"].to_numpy() - actual)))

    model = lgb.LGBMRegressor(random_state=42, verbose=-1, n_jobs=1)
    model.fit(train_feat[feature_cols], train_feat["target"])
    lightgbm_mae = float(np.mean(np.abs(model.predict(test_feat[feature_cols]) - actual)))

    return {"mae_naive": naive_mae, "mae_lightgbm": lightgbm_mae}


baseline_rows = []
for cust_id in baseline_ids:
    df = customers[cust_id][["dateTime", "net_load"]]
    split_at = df["dateTime"].max() - pd.Timedelta(days=90)
    result = compare_naive_and_lightgbm(df, split_at)
    if result is None:
        continue
    result["customer_id"] = cust_id
    baseline_rows.append(result)

baseline_results = pd.DataFrame(baseline_rows).set_index("customer_id")
baseline_results["lightgbm_beats_naive"] = baseline_results["mae_lightgbm"] < baseline_results["mae_naive"]
baseline_results.describe().round(3)

,mae_naive,mae_lightgbm
count,30.000,30.000
mean,0.764,0.640
std,0.648,0.517
min,0.050,0.055
25%,0.375,0.306
50%,0.536,0.509
75%,0.862,0.713
max,2.677,2.076


In [ ]:
win_rate = baseline_results["lightgbm_beats_naive"].mean() * 100
print(f"LightGBM beats naive persistence on {win_rate:.0f}% of the real sampled customers")

joined = forecastability.join(baseline_results, how="inner")
joined[["hurst_exponent", "permutation_entropy", "lightgbm_beats_naive"]].groupby("lightgbm_beats_naive").mean().round(
    3
)

LightGBM beats naive persistence on 87% of the real sampled customers


,hurst_exponent,permutation_entropy
lightgbm_beats_naive,,
False,0.188,0.978
True,0.611,0.993


## The PV signal hiding in net load

The vendored dataset carries no PV or EV sub-metering, so generation cannot be characterized as its own signal at full scale here. Two real, if indirect, proxies stand in for it instead: the meter's own real export activity (`activePowerIn`, checked directly against real winter/summer hourly averages to confirm it is the customer's feed into the grid, not consumption), and regional solar irradiance. Net load with rising DER adoption does not necessarily just get more volatile either, joint EV and PV activity can offset each other through a compensatory daytime effect elsewhere in the literature, worth checking directly here rather than assumed.

In [ ]:
weather_files = sorted(WEATHER_DIR.glob("*.parquet"))
weather = pd.read_parquet(weather_files[0])[["dateTime", "solarradiation"]]
print(f"regional weather proxy: {weather_files[0].name}, {weather['dateTime'].min()} to {weather['dateTime'].max()}")
print("one representative regional weather file stands in for a full per-community join here, a real simplification")

export_customers = {cust_id: df for cust_id, df in customers.items() if (df["activePowerIn"] > 0).mean() > 0.01}
print(f"customers with real, non-trivial export activity: {len(export_customers)} / {len(customers)}")

regional weather proxy: 02d3e508-f0a2-5f41-b52f-77c5d6f8f49d.parquet, 2022-08-28 23:00:00 to 2024-09-30 23:00:00
one representative regional weather file stands in for a full per-community join here, a real simplification
customers with real, non-trivial export activity: 5 / 300


In [ ]:
from twiga.core.stats.xicorr import compute_xicorr

xicorr_rows = []
for cust_id, df in export_customers.items():
    merged = df[["dateTime", "reactivePowerIn"]].merge(weather, on="dateTime", how="inner")
    if len(merged) < 100 or merged["reactivePowerIn"].std() == 0:
        continue
    result = compute_xicorr(merged, target_col="reactivePowerIn", variable_cols=["solarradiation"])
    xicorr_rows.append({"customer_id": cust_id, "xicorr_solar_export": result["xicor"].iloc[0]})

xicorr_export = pd.DataFrame(xicorr_rows).set_index("customer_id")
xicorr_export.describe().round(3)

,xicorr_solar_export
count,5.000
mean,0.366
std,0.214
min,0.027
25%,0.296
50%,0.437
75%,0.503
max,0.565


## From diagnostic to decision

A single recommendation that claims to fit every customer is a false confidence. The table below is a real, checkable mapping instead: which measured forecastability regime a customer falls into, and whether a trained model has actually earned its cost for that regime on this book's own data, not assumed to hold for everyone.

In [ ]:
decision = joined.copy()
decision["regime"] = np.select(
    [decision["hurst_exponent"] > 0.6, decision["permutation_entropy"] < 0.6],
    ["persistent (baseline likely enough)", "regular (baseline likely enough)"],
    default="volatile / low-persistence (LightGBM more likely to earn its cost)",
)

regime_summary = (
    decision.groupby("regime")
    .agg(
        customers=("regime", "size"),
        lightgbm_win_rate=("lightgbm_beats_naive", "mean"),
        mean_hurst=("hurst_exponent", "mean"),
    )
    .reset_index()
)
regime_summary["lightgbm_win_rate"] = (regime_summary["lightgbm_win_rate"] * 100).round(0)
regime_summary["mean_hurst"] = regime_summary["mean_hurst"].round(3)

themed_gt(
    GT(regime_summary)
    .tab_header(
        title=md("**Which Approach Earns Its Cost, By Regime**"),
        subtitle="A real confidence set, not one recommendation",
    )
    .cols_label(
        regime=md("**Forecastability regime**"),
        customers=md("**Customers**"),
        lightgbm_win_rate=md("**LightGBM win rate (%)**"),
        mean_hurst=md("**Mean Hurst**"),
    )
    .tab_source_note(f"n = {len(decision)} real customers with both a forecastability read and a baseline comparison"),
    n_rows=len(regime_summary),
)

GT(_tbl_data=                                              regime  customers  \
0                persistent (baseline likely enough)         22   
1  volatile / low-persistence (LightGBM more like...          8   

   lightgbm_win_rate  mean_hurst  
0              100.0       0.721  
1               50.0       0.097  , _body=<great_tables._gt_data.Body object at 0x12dcbc6e0>, _boxhead=Boxhead([ColInfo(var='regime', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Forecastability regime**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='lightgbm_win_rate', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**LightGBM win rate (%)**'), column_align='right', column_width=None), ColInfo(var='mean_hurst', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Mean Hurst**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x114711610>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Which Approach Earns Its Cost, By Regime**'), subtitle='A real confidence set, not one recommendation', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12dcb1910>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12dd2c740>, _source_notes=['n = 30 real customers with both a forecastability read and a baseline comparison'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='regime', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='lightgbm_win_rate', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mean_hurst', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='regime', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='customers', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='lightgbm_win_rate', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=